# MVP - Análise Técnica: Estratégia de Cruzamento de Médias (SMA)

## 1. Introdução e Objetivo

Este notebook documenta o processo de validação de uma estratégia técnica clássica para o projeto **FinSense**. 
O objetivo é realizar um *Backtest* (teste histórico) para verificar se a estratégia de **Cruzamento de Médias Móveis (SMA Crossover)** supera a simples manutenção do ativo (**Buy & Hold**) em um período de 5 anos.

**Metodologia Simplificada:**
- **Sem divisão de Treino/Teste:** Analisaremos a performance no período integral.
- **Ativo Base:** PETR4.SA (Petrobras PN).
- **Indicadores:** SMA Curta (20 períodos) e SMA Longa (50 períodos).

In [ ]:
# Instalação e Importação de Bibliotecas
# %pip install yfinance pandas matplotlib numpy

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configurações de plotagem
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 7)

## 2. Coleta e Tratamento de Dados

Utilizaremos a biblioteca `yfinance` para baixar dados diários ajustados.

In [ ]:
TICKER = 'PETR4.SA'
PERIOD = '5y'

# 1. Baixar Dados
print(f"Baixando dados para {TICKER}...")
df = yf.download(TICKER, period=PERIOD, progress=False)

# 2. Tratamento de MultiIndex (comum em versões recentes do yfinance)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# 3. Garantir apenas colunas essenciais
df = df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()

# 4. Calcular Retornos Diários do Ativo
df['Returns'] = df['Close'].pct_change()

print(f"Dados carregados: {len(df)} pregões.")
df.head()

## 3. Lógica da Estratégia (Engine)

Implementação do algoritmo de decisão:
1.  **Cálculo das Médias:** Simples (SMA) de 20 e 50 dias.
2.  **Sinal de Compra (1):** Quando a média curta (20) está acima da longa (50).
3.  **Sinal de Venda/Neutro (0):** Caso contrário.
4.  **Lag de Execução (Shift):** 
    *   *Importante:* Se o sinal é gerado no fechamento de hoje, só conseguimos operar na abertura (ou fechamento) de amanhã.
    *   Aplicamos `shift(1)` no sinal para alinhar a decisão tomada com o retorno do dia seguinte, evitando viés de futuro (*look-ahead bias*).

In [ ]:
# Parâmetros
SMA_SHORT = 20
SMA_LONG = 50

# 1. Calcular Indicadores
df['SMA_Short'] = df['Close'].rolling(window=SMA_SHORT).mean()
df['SMA_Long'] = df['Close'].rolling(window=SMA_LONG).mean()

# 2. Gerar Sinais (1 = Comprado, 0 = Fora)
df['Signal'] = np.where(df['SMA_Short'] > df['SMA_Long'], 1.0, 0.0)

# 3. Calcular Retornos da Estratégia
# Multiplicamos o sinal de ONTEM pelo retorno de HOJE
df['Strategy_Returns'] = df['Signal'].shift(1) * df['Returns']

# Limpeza de NaNs gerados pelo rolling windows
df.dropna(inplace=True)

df[['Close', 'SMA_Short', 'SMA_Long', 'Signal', 'Strategy_Returns']].tail()

## 4. Cálculo de Performance (Comparativo)

Compararemos a evolução patrimonial de R\$ 1.000,00 investidos.

In [ ]:
# 1. Curvas de Capital Acumuladas
df['Cumulative_Strategy'] = (1 + df['Strategy_Returns']).cumprod()
df['Cumulative_BuyHold'] = (1 + df['Returns']).cumprod()

# 2. Função de Métricas
def calculate_metrics(series):
    total_return = series.iloc[-1] - 1
    days = len(series)
    cagr = (1 + total_return) ** (252 / days) - 1
    volatility = series.pct_change().std() * np.sqrt(252)
    sharpe = cagr / volatility if volatility > 0 else 0
    
    # Drawdown
    peak = series.cummax()
    drawdown = (series - peak) / peak
    max_drawdown = drawdown.min()
    
    return {
        'Retorno Total': f"{total_return:.2%}",
        'Volatilidade Anual': f"{volatility:.2%}",
        'Sharpe Ratio': f"{sharpe:.2f}",
        'Max Drawdown': f"{max_drawdown:.2%}"
    }

# 3. Tabela Comparativa
metrics_strategy = calculate_metrics(df['Cumulative_Strategy'])
metrics_bh = calculate_metrics(df['Cumulative_BuyHold'])

results_df = pd.DataFrame([metrics_strategy, metrics_bh], index=['Estratégia SMA', 'Buy & Hold'])
results_df

## 5. Visualização (Gráficos)

In [ ]:
plt.figure(figsize=(16, 12))

# --- GRÁFICO 1: Preço e Médias ---
plt.subplot(2, 1, 1)
plt.plot(df.index, df['Close'], label='Preço (Close)', color='white', alpha=0.5)
plt.plot(df.index, df['SMA_Short'], label=f'SMA {SMA_SHORT}', color='yellow', linewidth=1.5)
plt.plot(df.index, df['SMA_Long'], label=f'SMA {SMA_LONG}', color='magenta', linewidth=1.5)

# Marcadores de Compra/Venda
# Onde o sinal muda de 0 para 1 (Compra)
buys = df[df['Signal'].diff() == 1]
# Onde o sinal muda de 1 para 0 (Venda)
sells = df[df['Signal'].diff() == -1]

plt.scatter(buys.index, buys['Close'], marker='^', color='green', s=100, label='Compra', zorder=5)
plt.scatter(sells.index, sells['Close'], marker='v', color='red', s=100, label='Venda', zorder=5)

plt.title(f'Indicadores Técnicos - {TICKER}')
plt.legend()
plt.grid(True, alpha=0.2)

# --- GRÁFICO 2: Comparação de Rentabilidade ---
plt.subplot(2, 1, 2)
plt.plot(df.index, df['Cumulative_Strategy'], label='Estratégia (SMA)', color='#00E396', linewidth=2)
plt.plot(df.index, df['Cumulative_BuyHold'], label='Buy & Hold', color='#777', linestyle='--', linewidth=1.5)

plt.title('Curva de Patrimônio (Equity Curve)')
plt.axhline(1.0, color='white', linestyle=':', alpha=0.3)
plt.legend()
plt.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 6. Conclusão

A análise acima permite visualizar o comportamento da estratégia em diferentes ciclos de mercado.

- **Mercados de Tendência:** A estratégia SMA tende a capturar grandes movimentos de alta, protegendo capital em quedas longas (ficando "zerada" ou fora).
- **Mercados Laterais:** O "ruído" do mercado pode gerar sinais falsos frequentes, corroendo rentabilidade frente ao Buy & Hold.

Verifique na tabela se o **Sharpe Ratio** (retorno ajustado ao risco) e o **Drawdown** (queda máxima) justificam a utilização desta estratégia ativa versus o investimento passivo.